In [1]:
import pandas as pd
import numpy as np

RAW = "../data/raw"

df = pd.read_parquet('../data/processed/protein_features.parquet')
hol = pd.read_csv(f"{RAW}/holidays_events.csv", parse_dates=["date"])

print("Holidays file:", hol.shape)
print("\nColumns:", list(hol.columns))
print("\n", hol.head(15))

Holidays file: (350, 6)

Columns: ['date', 'type', 'locale', 'locale_name', 'description', 'transferred']

          date     type    locale    locale_name  \
0  2012-03-02  Holiday     Local          Manta   
1  2012-04-01  Holiday  Regional       Cotopaxi   
2  2012-04-12  Holiday     Local         Cuenca   
3  2012-04-14  Holiday     Local       Libertad   
4  2012-04-21  Holiday     Local       Riobamba   
5  2012-05-12  Holiday     Local           Puyo   
6  2012-06-23  Holiday     Local       Guaranda   
7  2012-06-25  Holiday  Regional       Imbabura   
8  2012-06-25  Holiday     Local      Latacunga   
9  2012-06-25  Holiday     Local        Machala   
10 2012-07-03  Holiday     Local  Santo Domingo   
11 2012-07-03  Holiday     Local      El Carmen   
12 2012-07-23  Holiday     Local        Cayambe   
13 2012-08-05  Holiday     Local     Esmeraldas   
14 2012-08-10  Holiday  National        Ecuador   

                      description  transferred  
0              Fundacion d

In [2]:
print("TYPE breakdown:")
print(hol["type"].value_counts())

print("\nLOCALE breakdown:")
print(hol["locale"].value_counts())

print("\nTRANSFERRED breakdown:")
print(hol["transferred"].value_counts())

print("\nCross-tab of type × locale:")
print(pd.crosstab(hol["type"], hol["locale"]))

print("\nDate range:", hol["date"].min().date(), "to", hol["date"].max().date())

TYPE breakdown:
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

LOCALE breakdown:
locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

TRANSFERRED breakdown:
transferred
False    338
True      12
Name: count, dtype: int64

Cross-tab of type × locale:
locale      Local  National  Regional
type                                 
Additional     11        40         0
Bridge          0         5         0
Event           0        56         0
Holiday       137        60        24
Transfer        4         8         0
Work Day        0         5         0

Date range: 2012-03-02 to 2017-12-26


In [3]:
# Trap 1: do multiple holidays land on the same date+locale? (row explosion risk)
dupes = hol.groupby(["date", "locale", "locale_name"]).size()
print("Date+locale combos with >1 holiday:", (dupes > 1).sum())
print("\nWorst offenders:")
print(dupes[dupes > 1].sort_values(ascending=False).head(10))

# Trap 2: do locale_name values actually match our city/state values?
print("\nRegional locale_names:", sorted(hol[hol["locale"]=="Regional"]["locale_name"].unique()))
print("\nOur states:", sorted(df["state"].unique().tolist()))

Date+locale combos with >1 holiday: 7

Worst offenders:
date        locale    locale_name
2012-12-24  National  Ecuador        2
2012-12-31  National  Ecuador        2
2014-12-26  National  Ecuador        2
2016-05-01  National  Ecuador        2
2016-05-07  National  Ecuador        2
2016-05-08  National  Ecuador        2
2016-07-24  Local     Guayaquil      2
dtype: int64

Regional locale_names: ['Cotopaxi', 'Imbabura', 'Santa Elena', 'Santo Domingo de los Tsachilas']

Our states: ['Azuay', 'Bolivar', 'Chimborazo', 'Cotopaxi', 'El Oro', 'Esmeraldas', 'Guayas', 'Imbabura', 'Loja', 'Los Rios', 'Manabi', 'Pastaza', 'Pichincha', 'Santa Elena', 'Santo Domingo de los Tsachilas', 'Tungurahua']


In [4]:
# Clean the holiday table before joining:

# Restrict to our modeling window
hol = hol[(hol["date"] >= "2015-01-01") & (hol["date"] <= "2017-08-15")].copy()
print("In window:", len(hol))

# Trap: transferred=True means the holiday did NOT happen that day
# (it was officially moved; a separate type='Transfer' row marks the real date)
hol = hol[hol["transferred"] == False].copy()

# Trap: 'Work Day' is the INVERSE of a holiday — a Saturday worked to offset a bridge
workdays = hol[hol["type"] == "Work Day"][["date"]].copy()
workdays["is_workday_makeup"] = True
hol = hol[hol["type"] != "Work Day"].copy()

# Separate Events (earthquake, World Cup) from actual holidays
events = hol[hol["type"] == "Event"].copy()
hol = hol[hol["type"] != "Event"].copy()

print("\nAfter cleaning:")
print(hol["type"].value_counts())
print("\nEvents in window:")
print(events[["date", "description"]].to_string(index=False))

In window: 168

After cleaning:
type
Holiday       92
Additional    20
Transfer       7
Bridge         2
Name: count, dtype: int64

Events in window:
      date         description
2015-05-10     Dia de la Madre
2015-11-27        Black Friday
2015-11-30        Cyber Monday
2016-04-16    Terremoto Manabi
2016-04-17  Terremoto Manabi+1
2016-04-18  Terremoto Manabi+2
2016-04-19  Terremoto Manabi+3
2016-04-20  Terremoto Manabi+4
2016-04-21  Terremoto Manabi+5
2016-04-22  Terremoto Manabi+6
2016-04-23  Terremoto Manabi+7
2016-04-24  Terremoto Manabi+8
2016-04-25  Terremoto Manabi+9
2016-04-26 Terremoto Manabi+10
2016-04-27 Terremoto Manabi+11
2016-04-28 Terremoto Manabi+12
2016-04-29 Terremoto Manabi+13
2016-04-30 Terremoto Manabi+14
2016-05-01 Terremoto Manabi+15
2016-05-02 Terremoto Manabi+16
2016-05-03 Terremoto Manabi+17
2016-05-04 Terremoto Manabi+18
2016-05-05 Terremoto Manabi+19
2016-05-06 Terremoto Manabi+20
2016-05-07 Terremoto Manabi+21
2016-05-08 Terremoto Manabi+22
2016-05-08   

In [5]:
# build the three-way holiday flag

# Split by locale — each joins on a different key
nat = hol[hol["locale"] == "National"][["date", "type"]].drop_duplicates(subset=["date"])
nat["hol_national"] = True

reg = hol[hol["locale"] == "Regional"][["date", "locale_name"]].drop_duplicates()
reg = reg.rename(columns={"locale_name": "state"})
reg["hol_regional"] = True

loc = hol[hol["locale"] == "Local"][["date", "locale_name"]].drop_duplicates()
loc = loc.rename(columns={"locale_name": "city"})
loc["hol_local"] = True

print("National holiday dates:", len(nat))
print("Regional (date, state) pairs:", len(reg))
print("Local (date, city) pairs:", len(loc))

# Join each on its own key
before = len(df)

df = df.merge(nat[["date", "hol_national"]], on="date", how="left")
df = df.merge(reg, on=["date", "state"], how="left")
df = df.merge(loc, on=["date", "city"], how="left")

# Fill non-matches with False
for c in ["hol_national", "hol_regional", "hol_local"]:
    df[c] = df[c].fillna(False).astype(bool)

df["is_holiday"] = df["hol_national"] | df["hol_regional"] | df["hol_local"]

print(f"\nRows before: {before:,} | after: {len(df):,}")
assert len(df) == before, "ROW EXPLOSION — dedup failed"
print("Row count preserved ✓")

print("\nHoliday coverage:")
print(f"  National: {df['hol_national'].mean()*100:.1f}%")
print(f"  Regional: {df['hol_regional'].mean()*100:.1f}%")
print(f"  Local:    {df['hol_local'].mean()*100:.1f}%")
print(f"  Any:      {df['is_holiday'].mean()*100:.1f}%")

National holiday dates: 46
Regional (date, state) pairs: 10
Local (date, city) pairs: 64

Rows before: 7,459,611 | after: 7,459,611
Row count preserved ✓

Holiday coverage:
  National: 4.8%
  Regional: 0.0%
  Local:    0.4%
  Any:      5.2%


In [6]:
# Is it truly zero, or just rounding?
print("Regional holiday rows:", df["hol_regional"].sum())

# Which states do our stores actually sit in?
print("\nStore count by state:")
print(df.groupby("state", observed=True)["store_nbr"].nunique().sort_values(ascending=False))

# The 10 regional pairs
print("\nRegional holiday pairs:")
print(reg.to_string(index=False))

Regional holiday rows: 2207

Store count by state:
state
Pichincha                         19
Guayas                            11
Manabi                             3
Azuay                              3
Santo Domingo de los Tsachilas     3
Los Rios                           2
El Oro                             2
Cotopaxi                           2
Tungurahua                         2
Esmeraldas                         1
Bolivar                            1
Chimborazo                         1
Imbabura                           1
Loja                               1
Pastaza                            1
Santa Elena                        1
Name: store_nbr, dtype: int64

Regional holiday pairs:
      date                          state  hol_regional
2015-04-01                       Cotopaxi          True
2015-06-25                       Imbabura          True
2015-11-06 Santo Domingo de los Tsachilas          True
2015-11-07                    Santa Elena          True
2016-04-01      

In [9]:
# All dates any store treats as a holiday, by store
hol_dates = df[df["is_holiday"]][["date", "store_nbr"]].drop_duplicates()

# For each store, the sorted list of its holiday dates
store_hols = hol_dates.groupby("store_nbr")["date"].apply(lambda s: np.sort(s.unique()))

def days_to_next_holiday(row_dates, hol_array):
    """For each date, days until the next holiday (capped at 30)."""
    idx = np.searchsorted(hol_array, row_dates, side="left")
    idx = np.clip(idx, 0, len(hol_array) - 1)
    return np.clip((hol_array[idx] - row_dates).astype("timedelta64[D]").astype(int), 0, 30)

def days_from_prev_holiday(row_dates, hol_array):
    """For each date, days since the previous holiday (capped at 30)."""
    idx = np.searchsorted(hol_array, row_dates, side="right") - 1
    idx = np.clip(idx, 0, len(hol_array) - 1)
    return np.clip((row_dates - hol_array[idx]).astype("timedelta64[D]").astype(int), 0, 30)

to_hol, from_hol = np.zeros(len(df), dtype="int8"), np.zeros(len(df), dtype="int8")

for store, hol_array in store_hols.items():
    mask = (df["store_nbr"] == store).values
    dates = df.loc[mask, "date"].values
    to_hol[mask] = days_to_next_holiday(dates, hol_array)
    from_hol[mask] = days_from_prev_holiday(dates, hol_array)

df["days_to_holiday"] = to_hol
df["days_from_holiday"] = from_hol

print(df[["date", "store_nbr", "is_holiday", "days_to_holiday", "days_from_holiday"]].head(15))

         date  store_nbr  is_holiday  days_to_holiday  days_from_holiday
0  2015-01-31          1       False               16                  0
1  2015-02-01          1       False               15                  0
2  2015-02-02          1       False               14                  0
3  2015-02-03          1       False               13                  0
4  2015-02-04          1       False               12                  0
5  2015-02-05          1       False               11                  0
6  2015-02-06          1       False               10                  0
7  2015-02-07          1       False                9                  0
8  2015-02-08          1       False                8                  0
9  2015-02-09          1       False                7                  0
10 2015-02-10          1       False                6                  0
11 2015-02-11          1       False                5                  0
12 2015-02-12          1       False               

In [11]:
def days_from_prev_holiday(row_dates, hol_array):
    """Days since previous holiday (capped at 30). Returns 30 if none precedes."""
    idx = np.searchsorted(hol_array, row_dates, side="right") - 1
    no_prev = idx < 0                      # date precedes every holiday
    idx_safe = np.clip(idx, 0, len(hol_array) - 1)
    days = (row_dates - hol_array[idx_safe]).astype("timedelta64[D]").astype(int)
    days = np.clip(days, 0, 30)
    days[no_prev] = 30                     # treat as "far from any holiday"
    return days

# Recompute
to_hol, from_hol = np.zeros(len(df), dtype="int8"), np.zeros(len(df), dtype="int8")

for store, hol_array in store_hols.items():
    mask = (df["store_nbr"] == store).values
    dates = df.loc[mask, "date"].values
    to_hol[mask] = days_to_next_holiday(dates, hol_array)
    from_hol[mask] = days_from_prev_holiday(dates, hol_array)

df["days_to_holiday"] = to_hol
df["days_from_holiday"] = from_hol

# Verify across a holiday boundary
check = df[(df["store_nbr"] == 1) & (df["date"].between("2015-02-14", "2015-02-24"))]
print(check[["date", "is_holiday", "days_to_holiday", "days_from_holiday"]].to_string(index=False))

print("\ndays_from_holiday distribution:")
print(df["days_from_holiday"].value_counts().sort_index().head(10).drop_duplicates(subset=["date"]))

      date  is_holiday  days_to_holiday  days_from_holiday
2015-02-14       False                2                 30
2015-02-15       False                1                 30
2015-02-16        True                0                  0
2015-02-17        True                0                  0
2015-02-18       False               30                  1
2015-02-19       False               30                  2
2015-02-20       False               30                  3
2015-02-21       False               30                  4
2015-02-22       False               30                  5
2015-02-23       False               30                  6
2015-02-24       False               30                  7
2015-02-14       False                2                 30
2015-02-15       False                1                 30
2015-02-16        True                0                  0
2015-02-17        True                0                  0
2015-02-18       False               30                 

TypeError: Series.drop_duplicates() got an unexpected keyword argument 'subset'

In [12]:
# Event features (earthquake and shopping events):

# Earthquake: 31 consecutive days of severe demand distortion
eq = events[events["description"].str.startswith("Terremoto")][["date"]].copy()
eq["is_earthquake"] = True
df = df.merge(eq, on="date", how="left")
df["is_earthquake"] = df["is_earthquake"].fillna(False).astype(bool)

# Other national events (Mother's Day, Black Friday, Cyber Monday)
other_ev = events[~events["description"].str.startswith("Terremoto")][["date", "description"]].copy()
other_ev = other_ev.drop_duplicates(subset=["date"])
other_ev["is_event"] = True
df = df.merge(other_ev[["date", "is_event"]], on="date", how="left")
df["is_event"] = df["is_event"].fillna(False).astype(bool)

# Work Day: the inverse of a holiday — a Saturday worked to offset a bridge
if len(workdays) > 0:
    df = df.merge(workdays, on="date", how="left")
    df["is_workday_makeup"] = df["is_workday_makeup"].fillna(False).astype(bool)
else:
    df["is_workday_makeup"] = False

print("Row count:", len(df))
print(f"Earthquake rows: {df['is_earthquake'].sum():,} ({df['is_earthquake'].mean()*100:.2f}%)")
print(f"Event rows: {df['is_event'].sum():,}")
print(f"Workday-makeup rows: {df['is_workday_makeup'].sum():,}")

Row count: 7459611
Earthquake rows: 258,920 (3.47%)
Event rows: 56,638
Workday-makeup rows: 8,439


In [13]:
# verify the earthquake actually distorted demand, then save:

# Did protein demand genuinely spike?
eq_period = df[df["is_earthquake"]]["unit_sales"].mean()
normal = df[~df["is_earthquake"]]["unit_sales"].mean()
print(f"Mean sales during earthquake: {eq_period:.2f}")
print(f"Mean sales normally:          {normal:.2f}")
print(f"Difference: {(eq_period/normal - 1)*100:+.1f}%")

# Holiday effect check — the payoff for all this work
print("\nMean sales by days_to_holiday (0-7):")
print(df[df["days_to_holiday"] <= 7].groupby("days_to_holiday")["unit_sales"].mean().round(2))

df.to_parquet('../data/processed/protein_features_hol.parquet', index=False)
print("\nSaved:", df.shape)

Mean sales during earthquake: 7.63
Mean sales normally:          7.81
Difference: -2.3%

Mean sales by days_to_holiday (0-7):
days_to_holiday
0    8.34
1    7.58
2    7.77
3    7.42
4    6.82
5    8.01
6    7.92
7    8.45
Name: unit_sales, dtype: float32

Saved: (7459611, 45)


In [14]:
# 1. Earthquake effect in the affected province only
manabi = df[df["state"] == "Manabi"]
print("Manabi only:")
print(f"  During earthquake: {manabi[manabi['is_earthquake']]['unit_sales'].mean():.2f}")
print(f"  Normal:            {manabi[~manabi['is_earthquake']]['unit_sales'].mean():.2f}")
print(f"  Difference: {(manabi[manabi['is_earthquake']]['unit_sales'].mean()/manabi[~manabi['is_earthquake']]['unit_sales'].mean()-1)*100:+.1f}%")

# 2. Holiday effect controlling for day of week
print("\nMean sales by days_to_holiday, within Saturdays only:")
sat = df[df["dayofweek"] == 5]
print(sat[sat["days_to_holiday"] <= 7].groupby("days_to_holiday")["unit_sales"].mean().round(2))

# 3. Is the holiday itself different?
print("\nHoliday vs non-holiday:")
print(f"  Holiday:     {df[df['is_holiday']]['unit_sales'].mean():.2f}")
print(f"  Non-holiday: {df[~df['is_holiday']]['unit_sales'].mean():.2f}")

Manabi only:
  During earthquake: 5.25
  Normal:            4.42
  Difference: +18.9%

Mean sales by days_to_holiday, within Saturdays only:
days_to_holiday
0     9.04
1     8.66
2     8.94
3     6.66
4     7.62
5     6.77
6     9.00
7    10.03
Name: unit_sales, dtype: float32

Holiday vs non-holiday:
  Holiday:     8.36
  Non-holiday: 7.77
